In [ ]:
%matplotlib widget
import os
import numpy as np
import pandas as pd
import hyperspy.api as hs
import atomap.api as am
import atomap.initial_position_finding as ipf
from scipy.spatial.transform import Rotation as R
import matplotlib.pyplot as plt
import scipy
#import scipy.ndimage

import sidpy
from pyTEMlib.image_tools import decon_lr

In [ ]:
#Atomap-based algorithm is the following:
# - load STEM image 's'
# - create a copy with gaussian smooth applied 's1'
# - find atomic columnd by 's1' with manual tune of parameters
# - refine positions by center of mass and 2D gaussian using the original frame 's'
# - retrieve intensities
# - export the list of coordinates - with intensities and ellipticity - as a nunmpy array

In [ ]:
#Here we are loading STEM frame as TIFF file
#or any other format supported by HyperSpy 


folder = '/home/vasily/test_fit_atomap/'
fname = 'test_frame'

s = hs.load(folder+fname+'.tif')
#s = hs.load(folder+fname) #for dm3s


metadata = {}
metadata['fname'] = fname
#'''

#if one needs the scalebar, feel free to uncomment and add values
'''
imsize = (64,64)#nm
metadata['imsize'] = imsize
imsize_px = (s.axes_manager[0].size,s.axes_manager[1].size)
#xy directions not checked! has to be verified
d0,d1 = imsize[0]/imsize_px[0],imsize[1]/imsize_px[1]
print(d0,d1)
#Flaw!!! atomap apparently does not support non-sqare pixels!

s.axes_manager[0].scale = d0
s.axes_manager[1].scale = d1
s.axes_manager[0].units = 'nm'
s.axes_manager[1].units = 'nm'
'''

In [ ]:
#For combined BF/DF datasets
#s = s[1]

In [ ]:
s.plot()

In [ ]:
#Lucy-Richardson deconvolution (pyTEMlib) -- sharpen before column detection.
#decon_lr needs a sidpy Dataset; bridge from the hyperspy signal without transposing,
#so detection stays in the same [row, col] frame as the refinement. Square-pixel assumption.

def hs_to_sidpy(sig):
    arr = np.asarray(sig.data, dtype=float)
    d = sidpy.Dataset.from_array(arr)
    d.data_type = 'image'
    for i in (0, 1):
        sc = float(sig.axes_manager[i].scale)          # 1.0 if uncalibrated -> resolution in px
        un = sig.axes_manager[i].units
        un = un if isinstance(un, str) else 'px'        # hyperspy Undefined -> 'px'
        d.set_dimension(i, sidpy.Dimension(np.arange(arr.shape[i]) * sc,
            name=('x', 'y')[i], units=un,
            quantity='distance', dimension_type='spatial'))
    return d

resolution = 4.0      # axis units (px if uncalibrated) ~ column diameter; tune
lr = decon_lr(hs_to_sidpy(s), resolution=resolution, verbose=False)
s_decon = hs.signals.Signal2D(np.real(np.asarray(lr)))
s_decon.axes_manager[0].scale = s.axes_manager[0].scale
s_decon.axes_manager[1].scale = s.axes_manager[1].scale
s_decon.plot()

In [ ]:
#Detection image: light smooth of the deconvolution (tames LR noise for atomap).
#Refinement still uses the raw frame s.

s1 = s_decon.copy()
s1.map(scipy.ndimage.gaussian_filter, sigma=1)
s1.plot()
#fallback (pre-deconv behaviour): s1 = s.copy(); s1.map(scipy.ndimage.gaussian_filter, sigma=1)
#plt.close()

In [ ]:
#Preview

substract_backgound = True
threshold_rel = 0.25
pca = True

#Parameters of feature search should be the same in a preview (here)
#and in the actual run (next cell)

s_pks = am.get_feature_separation(s1, separation_range=(2, 20),subtract_background=substract_backgound,
                                  threshold_rel=threshold_rel, pca=pca, show_progressbar=False)
s_pks.plot()

In [ ]:
#Here we are manually picking the optimal value

sep = 5
atom_positions = am.get_atom_positions(s1,subtract_background=substract_backgound,
                                  threshold_rel=threshold_rel, pca=pca, separation=sep)
metadata['separation_1'] = sep

In [ ]:
#Opportunity for manual handling
atom_positions2 = ipf.add_atoms_with_gui(s1,atom_positions=atom_positions)
#atom_positions2 = ipf.add_atoms_with_gui(s,atom_positions=atom_positions2) #if manual rerun is needed

In [ ]:
#Preview of the outcome
plt.close('all')
sublattice = am.Sublattice(atom_positions2, image=s)
sublattice.plot()

In [ ]:
#Refine
sublattice.find_nearest_neighbors()
sublattice.refine_atom_positions_using_center_of_mass(s.data)

#Could be reasonable to tune the value of the % to the nearest neighbour threshold
#ptonn = 0.2
sublattice.refine_atom_positions_using_2d_gaussian(s.data)#percent_to_nn=ptonn
#metadata['percent_to_nn']=ptonn
#sublattice.refine_atom_positions_using_center_of_mass()

In [ ]:
sublattice.get_position_history().plot()

In [ ]:
#Preview of the output dataset
plt.close('all')
sublattice.plot()


In [ ]:
#Intensities are calculated here
i_points, i_record, p_record = sublattice.integrate_column_intensity()

In [ ]:
#Export -- canonical vmap interchange CSV (matches detect_columns.py: the same 14
#columns and %.8g precision, consumed by the lattice-refinement pipeline and as a start_csv seed).

df = pd.DataFrame({
    "x_obs0": sublattice.x_position, "y_obs0": sublattice.y_position,
    "ell0": np.asarray(sublattice.ellipticity) - 1,
    "rot0": -np.asarray(sublattice.rotation_ellipticity),
    "I_gauss": sublattice.atom_amplitude_gaussian2d, "I0": i_points,
    "sigma_x": sublattice.sigma_x, "sigma_y": sublattice.sigma_y,
    "x0_std":  [a.pixel_x_std for a in sublattice.atom_list],
    "y0_std":  [a.pixel_y_std for a in sublattice.atom_list],
    "sx_std":  [a.sigma_x_std for a in sublattice.atom_list],
    "sy_std":  [a.sigma_y_std for a in sublattice.atom_list],
    "rot_std": [a.rotation_std for a in sublattice.atom_list],
    "Ig_std":  [a.amplitude_gaussian_std for a in sublattice.atom_list],
})
df.to_csv(folder + fname.split('.')[0] + "_xyI.csv", index=False, float_format="%.8g")

In [ ]:
#Ellipticity check, if needed

In [ ]:
sublattice.plot_ellipticity_map(cmap='viridis', vmin=0.95, vmax=1.3)

In [ ]:
sublattice.plot_ellipticity_vectors()